# 06 - Evaluation and Baseline Comparison

This notebook compares the final human-reviewed assignment with the pre-review proposal and the simpler baselines.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

current = Path.cwd().resolve()
candidate_roots = [current, current.parent, current.parent.parent]
ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'smartassign_pipeline.py').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError('Could not locate the repository root.')

sys.path.insert(0, str(ROOT / 'src'))
from smartassign_pipeline import (
    FIGURES_DIR,
    MODELS_DIR,
    RESULTS_DIR,
    ModelBundle,
    build_score_and_uncertainty_matrices,
    compute_department_capacities,
    compute_department_profiles,
    current_assignment,
    ensure_output_directories,
    get_counterfactual_profile_columns,
    greedy_assignment,
    load_raw_data,
    random_assignment,
    save_csv,
    solve_assignment_with_overrides,
    split_train_test,
)

ensure_output_directories()
pd.set_option('display.max_columns', None)

df = load_raw_data()
bundle = ModelBundle.load(MODELS_DIR / 'best_model_bundle.joblib')
train_x, test_x, train_y, test_y = split_train_test(df, bundle.feature_columns)
sample_df = df.loc[test_x.index].copy().sample(n=min(50, len(test_x)), random_state=42)
profile_columns = get_counterfactual_profile_columns(bundle.feature_set_name, df)
department_profiles = compute_department_profiles(df, profile_columns)
score_matrix, uncertainty_matrix = build_score_and_uncertainty_matrices(bundle, sample_df, department_profiles, profile_columns)
capacities = compute_department_capacities(df, len(sample_df))

proposed_path = RESULTS_DIR / 'proposed_assignment.csv'
final_path = RESULTS_DIR / 'optimal_assignment.csv'
review_queue_path = RESULTS_DIR / 'review_queue.csv'
review_log_path = RESULTS_DIR / 'human_review_log.csv'

if proposed_path.exists():
    proposed_assignment = pd.read_csv(proposed_path)
else:
    proposed_assignment, _ = solve_assignment_with_overrides(score_matrix, sample_df.index, score_matrix.columns, capacity=capacities)

if final_path.exists():
    final_assignment = pd.read_csv(final_path)
else:
    final_assignment = proposed_assignment.copy()

review_queue = pd.read_csv(review_queue_path) if review_queue_path.exists() else pd.DataFrame(columns=['employee_index', 'department', 'predicted_score', 'uncertainty'])
review_log = pd.read_csv(review_log_path) if review_log_path.exists() else pd.DataFrame(columns=['timestamp', 'employee_id', 'department', 'action', 'reason', 'predicted_score', 'uncertainty'])

methods = {
    'ML+Optimize without review': proposed_assignment,
    'ML+Optimize + HITL': final_assignment,
    'Random': random_assignment(score_matrix, capacities, random_state=42),
    'Greedy': greedy_assignment(sample_df, score_matrix, capacities),
    'Current assignment': current_assignment(sample_df, score_matrix),
}

proxy_true_total = float(sample_df['Performance_Score'].sum())
comparison_rows = []
for method_name, assignment in methods.items():
    comparison_rows.append({
        'method': method_name,
        'total_predicted_score': float(assignment['predicted_score'].sum()),
        'mean_predicted_score': float(assignment['predicted_score'].mean()),
        'proxy_true_total_score': proxy_true_total,
    })
comparison = pd.DataFrame(comparison_rows).sort_values('total_predicted_score', ascending=False).reset_index(drop=True)
save_csv(comparison, RESULTS_DIR / 'assignment_comparison.csv')
display(comparison)

reviewed_pct = (len(review_queue) / len(proposed_assignment) * 100) if len(proposed_assignment) else 0.0
override_count = int((review_log['action'].fillna('') != 'Approve').sum()) if not review_log.empty else 0
override_pct = (override_count / len(review_queue) * 100) if len(review_queue) else 0.0
proposed_total = float(proposed_assignment['predicted_score'].sum())
final_total = float(final_assignment['predicted_score'].sum())

summary_table = pd.DataFrame([
    {
        'reviewed_pairs_pct': reviewed_pct,
        'overridden_pairs_pct': override_pct,
        'proposed_total_predicted_score': proposed_total,
        'final_total_predicted_score': final_total,
        'score_change_after_review': final_total - proposed_total,
    }
])
display(summary_table)
save_csv(summary_table, RESULTS_DIR / 'hitl_summary.csv')
display(Markdown('### Baseline comparison and review stats are shown above.'))